# Task 1 · Innovation — Named entities in Peruvian news

**Guiding question:** Which people, organizations and places dominate each news category?

Whereas Parts 1 and 2 classified headlines into thematic categories, this complementary analysis examines *who* and *what* the news is about. To this end, a named entity recognition (NER) model is applied to the 383 news items, and the most frequent entities are identified within each category. Accordingly, the analysis offers a substantive reading of the corpus that complements the classifier: it reveals which actors structure the agenda of each section and, in doing so, helps explain the vocabulary that the model relies upon.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from transformers import pipeline

This cell loads the libraries required for the analysis. Specifically, `transformers` provides access to the pretrained NER model, whereas `pandas` organizes the extracted entities and `plotly` renders the interactive charts.

In [2]:
# =============================================
#  CONFIGURATION
# =============================================

# Interactive figure in VS Code plus a static PNG copy that GitHub can display.
pio.renderers.default = "plotly_mimetype+png"

modelo_ner = "mrm8488/bert-spanish-cased-finetuned-ner"

# Only entities detected with high confidence are kept, to reduce false positives.
umbral_confianza = 0.80

# People, organizations and locations; the residual MISC type is excluded.
tipos_relevantes = ["PER", "ORG", "LOC"]

noticias = pd.read_csv("datos/noticias.csv")
print(noticias.shape)

(383, 7)


The configuration cell centralizes the parameters of the analysis. The model `mrm8488/bert-spanish-cased-finetuned-ner`, suggested by the assignment, is a Spanish BERT fine-tuned to recognize people (`PER`), organizations (`ORG`), locations (`LOC`) and miscellaneous entities (`MISC`). Since the latter type groups heterogeneous items, only the first three are retained. Furthermore, a confidence threshold of 0.80 is applied in order to discard uncertain detections, thereby favoring precision over recall.

## 1. Entity extraction

The NER model is applied to the headline and the summary of each news item. Combining both fields increases the number of entities detected, given that headlines often mention actors only by their surname, whereas summaries tend to identify them in full.

In [3]:
# "simple" aggregation merges the pieces of multi-word entities, e.g. "Keiko" + "Fujimori".
reconocedor = pipeline("ner", model=modelo_ner, aggregation_strategy="simple")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

c:\Users\Ana Karen\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ana Karen\.cache\huggingface\hub\models--mrm8488--bert-spanish-cased-finetuned-ner. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Ana Karen\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarnin

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  439MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

The NER pipeline is loaded with the `simple` aggregation strategy, which merges the individual tokens of a multi-word entity into a single item. Without this option, a name such as *Keiko Fujimori* would be split into separate fragments and counted incorrectly.

In [4]:
lista_entidades = []

for posicion, fila in noticias.iterrows():
    texto = f"{fila['titular']}. {fila['bajada']}"

    for entidad in reconocedor(texto):
        # The entity is read from the original text, which avoids broken sub-word fragments.
        nombre = texto[entidad["start"]:entidad["end"]].strip(" .,;:\"“”'‘’()")

        if entidad["score"] >= umbral_confianza and len(nombre) >= 3:
            lista_entidades.append({
                "etiqueta": fila["etiqueta"],
                "tipo": entidad["entity_group"],
                "entidad": nombre,
                "confianza": round(float(entidad["score"]), 3),
            })

    # Progress message every 100 news items.
    if posicion % 100 == 0:
        print(posicion, "news items processed")

0 news items processed
100 news items processed
200 news items processed
300 news items processed


The loop processes each news item and stores every entity that satisfies two conditions: a confidence of at least 0.80 and a minimum length of three characters, which filters out isolated initials and residual fragments. Importantly, each entity is recovered directly from the original text through its character positions and stripped of surrounding punctuation, which yields clean names regardless of how the model segments words internally.

In [5]:
entidades = pd.DataFrame(lista_entidades)
entidades = entidades[entidades["tipo"].isin(tipos_relevantes)]

frecuencias = (
    entidades.groupby(["etiqueta", "tipo", "entidad"])
    .size()
    .reset_index(name="frecuencia")
    .sort_values(["etiqueta", "frecuencia"], ascending=[True, False])
)
frecuencias.to_csv("datos/entidades.csv", index=False)

print("Entity mentions :", entidades.shape[0])
print("Unique entities :", frecuencias.shape[0])
entidades["tipo"].value_counts()

Entity mentions : 1413
Unique entities : 717


tipo
ORG    774
PER    390
LOC    249
Name: count, dtype: int64

The detected entities are gathered into a DataFrame, restricted to the three relevant types, and subsequently aggregated by category, type and name, so as to obtain the frequency of each entity within each category. The resulting table is exported to `datos/entidades.csv`, which allows the dashboard to display the entities without running the NER model again.